In [1]:
!pip install agentpy ipywidgets matplotlib seaborn
!jupyter labextension install @jupyter-widgets/jupyterlab-manager

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


(Deprecated) Installing extensions with the jupyter labextension install command is now deprecated and will be removed in a future major version of JupyterLab.

Users should manage prebuilt extensions with package managers like pip and conda, and extension authors are encouraged to distribute their extensions as prebuilt packages 
Building jupyterlab assets (production, minimized)


In [2]:
import agentpy as ap
import numpy as np
import random

# Agente: Carro
class DummyCar(ap.Agent):
    def setup(self):
        self.waiting = False  # Si está esperando en un semáforo rojo

    def step(self):
        env = self.model.environment
        x, y = env.positions[self]
        next_pos = env.next_position(x, y)
        traffic_light = env.traffic_light_at(next_pos)
        # Verifica si la siguiente posición está ocupada por otro carro
        occupied = any(pos == next_pos for car, pos in env.positions.items() if car in env.cars and car != self)
        # Se detiene si hay semáforo en rojo o si la posición está ocupada
        if (traffic_light and not traffic_light.is_green) or occupied:
            self.waiting = True
        else:
            self.waiting = False
            env.move_car(self, next_pos)

# Agente: Semáforo
class DummyTrafficLight(ap.Agent):
    def setup(self):
        self.is_green = random.choice([True, False])
        self.timer = 0

    def step(self):
        self.timer += 1
        # Cambia cada 5 pasos
        if self.timer % 5 == 0:
            self.is_green = not self.is_green

# Ambiente: Rotonda más grande
class RoundaboutEnvironment(ap.Grid):
    def setup(self):
        self.cars = self.model.cars
        self.traffic_lights = self.model.traffic_lights
        self.road_positions = [
            (1,3), (1,4), (2,5), (3,5), (4,5), (5,4), (5,3), (5,2),
            (4,1), (3,1), (2,1), (1,2)
        ]
        # Posiciones fijas para semáforos (ajustadas para 7x7)
        self.traffic_light_positions = [(2,2), (2,4), (4,2), (4,4)]
        for tl, pos in zip(self.traffic_lights, self.traffic_light_positions):
            self.add_agents([tl], positions=[pos])
        # Posiciones iniciales para carros (ajustadas para 7x7)
        for car, pos in zip(self.cars, self.road_positions):
            self.add_agents([car], positions=[pos])

    def next_position(self, x, y):
        idx = self.road_positions.index((x, y))
        return self.road_positions[(idx + 1) % len(self.road_positions)]
        
    def traffic_light_at(self, pos):
        for tl in self.traffic_lights:
            if self.positions[tl] == pos:
                return tl
        return None
    def move_car(self, car, new_pos):
        self.move_to(car, new_pos)

# Modelo
class RoundaboutModel(ap.Model):
    def setup(self):
        n_cars = self.p.n_cars
        n_lights = self.p.n_lights
        self.cars = ap.AgentList(self, n_cars, DummyCar)
        self.traffic_lights = ap.AgentList(self, n_lights, DummyTrafficLight)
        self.environment = RoundaboutEnvironment(self, (7,7))
        self.environment.setup()

    def step(self):
        self.traffic_lights.step()
        self.cars.step()

# Parámetros
parameters = {
    'n_cars': 4,
    'n_lights': 4,
    'steps': 20
}

model = RoundaboutModel(parameters)
results = model.run()

# Visualización (puedes adaptar la función de Act1 para mostrar el grid)

Completed: 20 steps
Run time: 0:00:00.006592
Simulation finished


In [8]:
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def roundabout_plot(model, ax):
    grid_size = model.environment.shape[0]
    ax.clear()
    ax.set_facecolor('white')

    # Parámetros de la rotonda y calles
    center = grid_size / 2
    roundabout_radius = 2  # Más pequeño para que se vean los carros completos
    street_width = 2.5

    # Dibuja las calles (Norte, Sur, Este, Oeste)
    # Calle vertical (Norte)
    rect_n = patches.Rectangle((center - street_width/2, 0), street_width, center - roundabout_radius, color='white', zorder=0)
    ax.add_patch(rect_n)
    # Calle vertical (Sur)
    rect_s = patches.Rectangle((center - street_width/2, center + roundabout_radius), street_width, center - roundabout_radius, color='white', zorder=0)
    ax.add_patch(rect_s)
    # Calle horizontal (Oeste)
    rect_w = patches.Rectangle((0, center - street_width/2), center - roundabout_radius, street_width, color='white', zorder=0)
    ax.add_patch(rect_w)
    # Calle horizontal (Este)
    rect_e = patches.Rectangle((center + roundabout_radius, center - street_width/2), center - roundabout_radius, street_width, color='white', zorder=0)
    ax.add_patch(rect_e)

    # Dibuja la rotonda (círculo negro en el centro)
    rotonda = patches.Circle((center, center), radius=roundabout_radius, color='black', fill=True, zorder=1)
    ax.add_patch(rotonda)

    # Dibuja los carros (puedes adaptar esto a tus posiciones)
    for car in model.cars:
        pos = model.environment.positions[car]
        ax.plot(pos[1], pos[0], 'ks', markersize=10, zorder=2)

    ax.set_xlim(0, grid_size)
    ax.set_ylim(0, grid_size)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title('Rotonda simple: círculo negro pequeño y cuatro calles')

model = RoundaboutModel(parameters)
fig, ax = plt.subplots(figsize=(7,7))
animation = ap.animate(model, fig, ax, roundabout_plot)
HTML(animation.to_jshtml())